In [ ]:
# EuroSAT Land Use Classification with EfficientNet-B3
# Simple, focused implementation with Gradio UI (matches resnet50.ipynb layout & fields)

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import requests
from io import BytesIO
import gradio as gr
import os

# ============================================================================
# SECTION 1: Setup and Configuration
# ============================================================================

# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")
    DRIVE_MOUNTED = True
except ImportError:
    print("ℹ️ Not running in Colab - using local file system")
    DRIVE_MOUNTED = False

# EuroSAT class names (same as in resnet50 notebook)
EUROSAT_CLASSES = [
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
]

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")

# ============================================================================
# SECTION 2: Model Architecture (EfficientNet-B3)
# ============================================================================

class EuroSATEfficientNetB3(nn.Module):
    """
    EfficientNet-B3 model customized for EuroSAT classification

    Architecture:
    - Backbone: EfficientNet-B3 pretrained on ImageNet (if available)
    - Classifier replaced to output num_classes
    - Total parameters: ~12M (depending on implementation)
    """
    def __init__(self, num_classes=10, pretrained=True):
        super(EuroSATEfficientNetB3, self).__init__()

        # Attempt to load torchvision EfficientNet-B3 with ImageNet weights if available
        try:
            # Newer torchvision uses models.EfficientNet_B3_Weights
            self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None)
        except Exception:
            # Fallback for older torchvision versions
            self.backbone = models.efficientnet_b3(pretrained=pretrained)

        # Replace classifier to match EuroSAT classes
        # torchvision EfficientNet typically uses backbone.classifier[1] as final fc
        try:
            in_features = self.backbone.classifier[1].in_features
            self.backbone.classifier[1] = nn.Linear(in_features, num_classes)
        except Exception:
            # Safe fallback: replace entire classifier
            self.backbone.classifier = nn.Sequential(
                nn.Dropout(p=0.4),
                nn.Linear(self.backbone.classifier[1].in_features if hasattr(self.backbone, 'classifier') else 1536, num_classes)
            )

    def forward(self, x):
        return self.backbone(x)

# ============================================================================
# SECTION 3: Image Preprocessing
# ============================================================================

def get_transforms(img_size=224):
    """
    Get image preprocessing transforms

    Transforms:
    1. Resize to img_size x img_size
    2. Convert to tensor
    3. Normalize with ImageNet mean and std
    """
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],  # ImageNet mean
            std=[0.229, 0.224, 0.225]     # ImageNet std
        )
    ])

# ============================================================================
# SECTION 4: Model Loading (robust)
# ============================================================================

def _normalize_state_dict_keys(state_dict):
    """
    Remove common prefixes like 'module.', 'model.', 'backbone.' from keys.
    """
    new_sd = {}
    for k, v in state_dict.items():
        new_k = k
        for prefix in ('module.', 'model.', 'backbone.', 'net.'):
            if new_k.startswith(prefix):
                new_k = new_k[len(prefix):]
        new_sd[new_k] = v
    return new_sd

def load_model(model_path):
    """
    Load EfficientNet-B3 model from checkpoint

    Args:
        model_path: Path to .pth model file (can be Drive-relative or absolute)

    Returns:
        model: Loaded model in eval mode or None on failure
        info: Dictionary with model information or error info
    """
    try:
        # Handle Google Drive paths (same logic as resnet50 notebook)
        if DRIVE_MOUNTED and not model_path.startswith('/content/drive/'):
            if not model_path.startswith('/'):
                model_path = '/content/drive/MyDrive/' + model_path

        print(f"📂 Loading model from: {model_path}")
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Checkpoint not found at: {model_path}")

        checkpoint = torch.load(model_path, map_location=device)

        # Create model
        model = EuroSATEfficientNetB3(num_classes=len(EUROSAT_CLASSES), pretrained=True)

        # Determine state dict from checkpoint
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                state_dict = checkpoint['model_state_dict']
                accuracy = checkpoint.get('test_acc', checkpoint.get('val_accuracy', 'N/A'))
                epoch = checkpoint.get('epoch', 'N/A')
            elif 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
                accuracy = checkpoint.get('test_acc', checkpoint.get('val_accuracy', 'N/A'))
                epoch = checkpoint.get('epoch', 'N/A')
            else:
                # Heuristic: if values look like tensors, assume checkpoint is a state_dict
                vals = list(checkpoint.values())
                if vals and torch.is_tensor(vals[0]):
                    state_dict = checkpoint
                    accuracy = 'N/A'
                    epoch = 'N/A'
                else:
                    # Try to find nested dicts
                    found = None
                    for v in checkpoint.values():
                        if isinstance(v, dict):
                            inner_vals = list(v.values())
                            if inner_vals and torch.is_tensor(inner_vals[0]):
                                found = v
                                break
                    state_dict = found if found is not None else checkpoint
                    accuracy = checkpoint.get('test_acc', checkpoint.get('val_accuracy', 'N/A')) if isinstance(checkpoint, dict) else 'N/A'
                    epoch = checkpoint.get('epoch', 'N/A') if isinstance(checkpoint, dict) else 'N/A'
        else:
            # checkpoint is likely a raw state_dict
            state_dict = checkpoint
            accuracy = 'N/A'
            epoch = 'N/A'

        # Try loading state_dict as-is
        try:
            model.load_state_dict(state_dict)
            load_msg = "Loaded checkpoint with strict matching."
        except Exception as e_strict:
            # Try normalized keys
            norm_sd = _normalize_state_dict_keys(state_dict)
            try:
                model.load_state_dict(norm_sd)
                load_msg = "Loaded checkpoint after normalizing keys (removed common prefixes)."
            except Exception as e_non_strict:
                # Try non-strict load (partial)
                try:
                    model.load_state_dict(norm_sd, strict=False)
                    load_msg = ("Partially loaded checkpoint (non-strict). Some keys were missing or mismatched; "
                                "please verify compatibility.")
                except Exception as final_err:
                    raise RuntimeError(f"Failed to load checkpoint into model.\nStrict error: {e_strict}\nNormalized error: {e_non_strict}\nFinal error: {final_err}")

        # Move to device & eval
        model.to(device)
        model.eval()

        # Model info
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        info = {
            'accuracy': accuracy,
            'epoch': epoch,
            'total_params': total_params,
            'trainable_params': trainable_params,
            'load_message': load_msg
        }

        print("✅ Model loaded successfully!")
        print(f"   {load_msg}")
        print(f"   Accuracy: {accuracy}")
        print(f"   Epoch: {epoch}")
        print(f"   Parameters: {total_params:,}")

        return model, info

    except Exception as e:
        print(f"❌ Error loading model: {str(e)}")
        return None, {'error': str(e)}

# ============================================================================
# SECTION 5: Prediction Function
# ============================================================================

def predict(model, image, transform):
    """
    Make prediction on an image

    Args:
        model: Loaded PyTorch model
        image: PIL Image
        transform: Image preprocessing transforms

    Returns:
        results: Dictionary with prediction results
    """
    try:
        image_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(image_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)

            probs = probabilities[0].cpu().numpy()
            predicted_idx = int(torch.argmax(outputs, dim=1).item())
            confidence = float(probabilities[0][predicted_idx].item())
            predicted_class = EUROSAT_CLASSES[predicted_idx]

            top5_indices = np.argsort(probs)[-5:][::-1]
            top5_classes = [EUROSAT_CLASSES[i] for i in top5_indices]
            top5_probs = [float(probs[i]) for i in top5_indices]

            return {
                'predicted_class': predicted_class,
                'confidence': confidence,
                'all_probabilities': probs,
                'top5_classes': top5_classes,
                'top5_probs': top5_probs
            }

    except Exception as e:
        print(f"❌ Prediction error: {str(e)}")
        return None

# ============================================================================
# SECTION 6: Visualization (same style as resnet50)
# ============================================================================

def create_bar_chart(results):
    """Create bar chart of top 5 predictions (horizontal)"""
    if results is None:
        return None

    fig, ax = plt.subplots(figsize=(10, 6))

    classes = results['top5_classes']
    probs = [p * 100 for p in results['top5_probs']]

    colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(classes))]
    bars = ax.barh(classes, probs, color=colors)

    # Add percentage labels
    for i, (bar, prob) in enumerate(zip(bars, probs)):
        ax.text(prob + 1, bar.get_y() + bar.get_height()/2,
                f'{prob:.1f}%', va='center', fontweight='bold')

    ax.set_xlabel('Confidence (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Land Use Class', fontsize=12, fontweight='bold')
    ax.set_title('Top 5 Predictions (EfficientNet-B3)', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 105)
    ax.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    return fig

# ============================================================================
# SECTION 7: Gradio Interface (mirrors ResNet50 UI fields & default paths)
# ============================================================================

# Global model variable
loaded_model = None
transform = get_transforms()

def load_model_gradio(model_path):
    """Load model for Gradio interface"""
    global loaded_model

    if not model_path or not model_path.strip():
        return "⚠️ Please provide a model path"

    loaded_model, info = load_model(model_path.strip())

    if loaded_model is None:
        return f"❌ Failed to load model: {info.get('error', 'Unknown error')}"

    return f"""✅ Model loaded successfully!

📊 Model Information:
• Accuracy: {info.get('accuracy', 'N/A')}
• Epoch: {info.get('epoch', 'N/A')}
• Total Parameters: {info.get('total_params', 0):,}
• Trainable Parameters: {info.get('trainable_params', 0):,}

{info.get('load_message', '')}

Ready to make predictions!"""

def predict_gradio(image):
    """Make prediction for Gradio interface"""
    global loaded_model

    if loaded_model is None:
        return None, "❌ Please load a model first", None

    if image is None:
        return None, "⚠️ Please provide an image", None

    results = predict(loaded_model, image, transform)

    if results is None:
        return None, "❌ Prediction failed", None

    # Create results text
    results_text = f"""✅ Prediction Complete!

🎯 Predicted Class: {results['predicted_class']}
📈 Confidence: {results['confidence']*100:.2f}%

📊 Top 5 Predictions:
"""
    for cls, prob in zip(results['top5_classes'], results['top5_probs']):
        results_text += f"• {cls}: {prob*100:.1f}%\n"

    chart = create_bar_chart(results)

    return image, results_text, chart

def load_from_url(url):
    """Load image from URL"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        return image
    except Exception as e:
        print(f"URL load error: {e}")
        return None

# Create Gradio interface (fields and layout mirror resnet50.ipynb)
def create_interface():
    with gr.Blocks(theme=gr.themes.Soft(), title="EuroSAT EfficientNet-B3 Classifier") as demo:

        gr.Markdown("""
        # 🛰️ EuroSAT Land Use Classification with EfficientNet-B3

        Upload a satellite or aerial image to classify it into one of 10 land use categories.

        **Classes**: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial,
        Pasture, PermanentCrop, Residential, River, SeaLake
        """)

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 🔧 Step 1: Load Model")

                # Default path mirrors the ResNet notebook style but for EfficientNet-B3
                model_path_input = gr.Textbox(
                    label="Model Path",
                    placeholder="/content/drive/MyDrive/weights/efficientnet_b3_eurosat_best.pth",
                    value="/content/drive/MyDrive/weights/efficientnet_b3_eurosat_best.pth",
                    info="Path to your EfficientNet-B3 model weights (.pth file)"
                )

                load_btn = gr.Button("📥 Load Model", variant="primary")
                model_status = gr.Textbox(
                    label="Model Status",
                    lines=8,
                    interactive=False
                )

                gr.Markdown("""
                ### 📝 Path Examples:
                - Google Drive: `/content/drive/MyDrive/weights/model.pth`
                - Relative: `weights/model.pth`
                - Local: `/path/to/model.pth`
                """)

            with gr.Column(scale=1):
                gr.Markdown("### 🖼️ Step 2: Upload Image")

                image_input = gr.Image(
                    label="Upload Image",
                    type="pil",
                    height=300
                )

                gr.Markdown("**Or load from URL:**")
                image_url = gr.Textbox(
                    label="Image URL",
                    placeholder="https://example.com/satellite-image.jpg"
                )
                load_url_btn = gr.Button("🔗 Load from URL")

                predict_btn = gr.Button("🔍 Classify Image", variant="primary", size="lg")

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📊 Results")
                results_text = gr.Textbox(
                    label="Prediction Results",
                    lines=12,
                    interactive=False
                )

            with gr.Column(scale=1):
                gr.Markdown("### 📈 Confidence Chart")
                results_chart = gr.Plot(label="Top 5 Predictions")

        gr.Markdown("""
        ---
        ### 📖 Instructions:
        1. **Load Model**: Enter the path to your trained EfficientNet-B3 model and click "Load Model"
        2. **Upload Image**: Either upload an image file or paste a URL
        3. **Classify**: Click "Classify Image" to see the prediction

        ### 🔬 Model Details:
        - **Architecture**: EfficientNet-B3 with replaced classifier head
        - **Input Size**: 224×224 pixels (auto-resized)
        - **Output**: 10 land use classes
        - **Preprocessing**: ImageNet normalization
        """)

        # Event handlers (same usage as resnet50 notebook)
        load_btn.click(
            fn=load_model_gradio,
            inputs=[model_path_input],
            outputs=[model_status]
        )

        load_url_btn.click(
            fn=load_from_url,
            inputs=[image_url],
            outputs=[image_input]
        )

        predict_btn.click(
            fn=predict_gradio,
            inputs=[image_input],
            outputs=[image_input, results_text, results_chart]
        )

    return demo

# ============================================================================
# SECTION 8: Launch
# ============================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("🛰️ EuroSAT LAND USE CLASSIFICATION - EfficientNet-B3")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Google Drive: {'✅ Mounted' if DRIVE_MOUNTED else '❌ Not mounted'}")
    print(f"Classes: {len(EUROSAT_CLASSES)}")
    print("=" * 70)

    demo = create_interface()
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True,
        debug=True
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully!
🔧 Using device: cpu
🛰️ EuroSAT LAND USE CLASSIFICATION - EfficientNet-B3
Device: cpu
Google Drive: ✅ Mounted
Classes: 10
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://83a1f1e92c0ef93bbf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📂 Loading model from: /content/drive/MyDrive/efficientnet_b3_eurosat_best.pth
✅ Model loaded successfully!
   Partially loaded checkpoint (non-strict). Some keys were missing or mismatched; please verify compatibility.
   Accuracy: N/A
   Epoch: N/A
   Parameters: 10,711,602
